# GHG Emissions Inventory - Silver Layer

## Objective
Transform, cleanse, and standardize raw greenhouse gas (GHG) emissions data from bronze.bronze_ghg_emissions to build a clean, production-ready Silver Delta table (silver.silver_ghg_emissions).

## Data Flow
bronze.bronze_ghg_emissions → Spark DataFrame → silver.silver_ghg_emissions

## Source
The underlying data comes from the Paris OpenData API: Inventaire des émissions de gaz à effet de serre du territoire.

## Input
Bronze Delta table: bronze.bronze_ghg_emissions

## Output
Silver Delta table: silver.silver_ghg_emissions

## Silver Layer Principle
The Silver layer cleanses, enforces schema integrity, and standardizes data structures for downstream analytics. Column names are translated from French to English, missing primary key records are handled, and numeric fields are explicitly cast.

## Processing Steps
1. **Load Bronze Data:** Read raw table into PySpark DataFrame.
2. **Inspect Schema:** Print schema and sample rows.
3. **Standardize Column Names:** Rename columns to English.
4. **Data Quality & Null Handling:** Check for nulls and filter out records without year.
5. **Data Type Conversions:** Cast year to DateType and metrics to DoubleType.
6. **Write to Silver:** Save cleaned DataFrame to silver.silver_ghg_emissions Delta table.
7. **SQL Verification:** Query silver table to validate row count and schema.

In [0]:
# Importing libraries
from pyspark.sql.functions import col, count, when

# LOAD BRONZE DATA

In [0]:

# Load data from bronze schema
df_bronze_ghg = spark.table("workspace.bronze.bronze_ghg_emission")

In [0]:
# Inspecting the schema 
df_bronze_ghg.printSchema()

In [0]:
# count number of rows in `df_bronze_ghg` 
df_count=df_bronze_ghg.count()
print(df_count)

In [0]:
# display the dataframe 
df_bronze_ghg.limit(10).display()

# CLEAN DATA  AND CHECK FOR NULLS

In [0]:
# Mapping French API column names to standardized English column names
column_mapping = {
    "annee": "year",
    "ges_grd_secteurs_total": "ghg_major_sectors_total",
    "ges_grd_secteurs_energie": "ghg_major_sectors_energy",
    "ges_grd_secteurs_transport": "ghg_major_sectors_transport",
    "ges_grd_secteurs_consommation": "ghg_major_sectors_consumption",
    "ges_detail_locales_total": "ghg_local_details_total",
    "ges_detail_locales_residentiel": "ghg_local_details_residential",
    "ges_detail_locales_tertiaire": "ghg_local_details_commercial",
    "ges_detail_locales_industrie": "ghg_local_details_industry",
    "ges_detail_locales_transport": "ghg_local_details_transport",
    "ges_detail_locales_dechets": "ghg_local_details_waste",
    "ges_detail_hors_paris_total": "ghg_outside_paris_details_total",
    "ges_detail_hors_paris_construction_materiaux": "ghg_outside_paris_details_construction_materials",
    "ges_detail_hors_paris_transport": "ghg_outside_paris_details_transport",
    "ges_detail_hors_paris_transport_aerien": "ghg_outside_paris_details_air_transport",
    "ges_detail_hors_paris_alimentation": "ghg_outside_paris_details_food",
    "ges_detail_hors_paris_amont_energie": "ghg_outside_paris_details_upstream_energy",
    "evolution": "evolution_percentage"
}


In [0]:
# Loop to dynamically rename columns in your PySpark DataFrame
for old_col, new_col in column_mapping.items():
    df_bronze_ghg = df_bronze_ghg.withColumnRenamed(old_col, new_col)

In [0]:
# Display the new column names 
df_bronze_ghg.display()

In [0]:
# Check for nulls
null_counts_df = df_bronze_ghg.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_bronze_ghg.columns
])

# Display the summary table showing NULL count per column
display(null_counts_df)

In [0]:
# Convert year column into integer
df_clean= df_bronze_ghg.withColumn("year", col("year").cast("integer"))

In [0]:
# Display the clean dataframe
df_clean.display()

# WRITE TO SILVER LAYER

In [0]:
# Write to silver 
df_clean \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.silver_ghg_emissions")

# CHECKING THE SILVER TABLE

In [0]:
%sql
SELECT * 
FROM workspace.silver.silver_ghg_emissions;